# 3-Class Forgery Classifier (Real / Edited / Deepfake)

Thin notebook -- logic lives in `data/*.py` and `model/*.py`. Sections: Data, Model, Train, Eval.
See `architecture_decisions.md` for the design and `data_download.md` / `model_code.md` for the implementation instructions this follows.

## Setup

In [24]:
# %cd ..
# %rm -r deepfake

In [1]:
!git clone https://github.com/satyagalla/deepfake.git
%cd deepfake

fatal: destination path 'deepfake' already exists and is not an empty directory.
/content/deepfake


In [18]:
!git pull

remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.58 KiB | 1.79 MiB/s, done.
From https://github.com/satyagalla/deepfake
   9179094..40b80a6  main       -> origin/main
Updating 9179094..40b80a6
Fast-forward
 forgery_classifier.ipynb |  16 ++++-
 model/demo.py            | 171 +++++++++++++++++++++++++++++++++++++++++++++++
 requirements.txt         |   1 +
 3 files changed, 187 insertions(+), 1 deletion(-)
 create mode 100644 model/demo.py


In [4]:
!pip install -q -r requirements.txt

In [5]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
# Confirm GPU tier per model_code.md section 0 -- A100 gets the full 3-branch
# config below as-is; T4/L4 need the fallback config from architecture_decisions.md's GPU-tier table.

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [7]:
# Mount Drive if running in Colab and you want raw/checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DEEPFAKE_DATA_ROOT'] = '/content/drive/MyDrive/deepfake'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# One-time: copy already-downloaded/face-filtered data from this session's local
# Colab disk (/content/deepfake/data_raw, dataset) to the Drive location config.py
# now points at -- so the download and face_filter cells above don't need to be
# re-run. Safe to re-run this cell: skips any folder that doesn't exist locally,
# and skips the copy entirely if the Drive destination already has content.
import shutil
from pathlib import Path

LOCAL_ROOT = Path('/content/deepfake')  # adjust if your repo isn't cloned here
DRIVE_ROOT = Path(os.environ['DEEPFAKE_DATA_ROOT'])

for name in ('data_raw', 'dataset', 'checkpoints'):
    src = LOCAL_ROOT / name
    dst = DRIVE_ROOT / name
    if not src.exists():
        print(f"skip {name}: no local {src}")
        continue
    if dst.exists() and any(dst.iterdir()):
        print(f"skip {name}: {dst} already has content")
        continue
    print(f"copying {src} -> {dst} ...")
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"done: {name}")

copying /content/deepfake/data_raw -> /content/drive/MyDrive/deepfake/data_raw ...
done: data_raw
copying /content/deepfake/dataset -> /content/drive/MyDrive/deepfake/dataset ...
done: dataset
copying /content/deepfake/checkpoints -> /content/drive/MyDrive/deepfake/checkpoints ...
done: checkpoints


## Data
Downloads the three raw sources, then runs uniform MTCNN face detect+crop, split, and manifest generation.

In [7]:
from data.download import test_sources

# Test cell: no-download validation of all three sources -- HF schema + person-caption
# hit rate for COCO_AI, Kaggle auth + file listing for CASIA and PS-Battles, and (for
# PS-Battles specifically) the originals-vs-derivatives folder heuristic checked against
# that listing before anything is downloaded. Catches a broken token, a changed HF
# schema, or an unverified PS-Battles layout in seconds instead of after paying for the
# full transfer. Raises SystemExit with a per-source breakdown if anything fails.
test_sources()

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[COCO_AI] streaming 200 rows to check schema + person-caption hit rate (no files written)...
[CASIA] Kaggle auth OK, 20 files listed for divg07/casia-20-image-tampering-detection-dataset (sample: ['CASIA2/Au/Au_ani_00001.jpg', 'CASIA2/Au/Au_ani_00002.jpg', 'CASIA2/Au/Au_ani_00003.jpg', 'CASIA2/Au/Au_ani_00004.jpg', 'CASIA2/Au/Au_ani_00005.jpg'])
[CASIA] path-heuristic counts across 20 image file(s) in this listing: {'other': 20}
[CASIA] listing looks paginated (>=20 files, dataset is likely larger) -- treating the heuristic counts above as informational only, not a hard pass/fail.
[CASIA] OK

[COCO_AI] schema OK -- 62/200 sampled rows are usable person-caption pairs (~31% hit rate)
[COCO_AI] OK

All sources OK -- safe to run the full download.


In [8]:
from data.download import download_all

# Final cell: full download, all three sources (COCO_AI, CASIA, PS-Battles) run
# concurrently -- independent I/O-bound network pulls, so total wall-clock is
# roughly the slowest of the three instead of the sum. Only run this after the
# test cell above passes.
download_all(n_pairs=3000) #change buffer_size to 500 for faster downloads in data\download.py line #69

Streaming NasrinImp/COCO_AI (train split), collecting 3000 person-caption pairs...
/content/deepfake/data_raw/edited_src already has files, skipping Kaggle download (use --force to redo).
[CASIA] done.


COCO_AI person pairs:   0%|          | 0/3000 [00:00<?, ?pair/s]

COCO_AI: scanned 10017 rows, wrote 2255 person-caption real/deepfake pairs to /content/deepfake/data_raw/real_src and /content/deepfake/data_raw/deepfake_src
[COCO_AI] done.


In [9]:
import subprocess, sys

# Test cell: quick diagnostic pass over 300 images/class -- writes to a separate
# dataset_diag/ dir (doesn't touch the real dataset), skips the deepfake
# survival-floor check, and prints per-class decode/mtcnn/save timing plus
# GPU-batch-fragmentation stats. Confirms MTCNN/CUDA and both edited sources
# (CASIA + PS-Battles, including the PS-Battles folder heuristic) are wired up
# correctly before paying for the full pass.
subprocess.run([sys.executable, 'data/face_filter.py', '--limit', '300'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'data/face_filter.py', '--limit', '300'], returncode=0)

In [10]:
import subprocess, sys

# Final cell: full run over every downloaded source. face_filter.py is a script
# (not just importable functions) -- run it directly so its per-source logging
# (seen/detected/rejected, timing, fragmentation) prints in full, including the
# deepfake survival-rate guard. Only run this after the test cell above passes.
subprocess.run([sys.executable, 'data/face_filter.py'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'data/face_filter.py'], returncode=0)

In [11]:
# Class balance check -- must feed the class-weighted loss below, never hardcoded proportions
import pandas as pd
from config import MANIFEST_PATH

manifest = pd.read_csv(MANIFEST_PATH)
print(manifest.groupby(['class', 'split']).size())

class     split
deepfake  train    1780
          val       313
edited    train     922
          val       162
real      train    1653
          val       291
dtype: int64


## Model
Instantiate the 3-branch fusion model and dry-run a single batch through the full pipeline before committing to the full training loop.

In [12]:
from config import DEVICE, EMBED_DIM, CLASSES
from model.dataset import get_dataloader
from model.fusion import ForgeryClassifier

model = ForgeryClassifier(embed_dim=EMBED_DIM, num_classes=len(CLASSES)).to(DEVICE)

dry_loader = get_dataloader('train', batch_size=8, shuffle=True, num_workers=2)
batch = next(iter(dry_loader))
rgb = batch['rgb'].to(DEVICE)
fft_mag = batch['fft_mag'].to(DEVICE)
srm = batch['srm_residual'].to(DEVICE)

logits, gate_weights = model(rgb, fft_mag, srm)
print('logits:', logits.shape, 'gate_weights:', gate_weights.shape)
assert logits.shape == (8, len(CLASSES))
assert gate_weights.shape == (8, 3)
print('Dry run OK -- shapes check out.')

logits: torch.Size([8, 3]) gate_weights: torch.Size([8, 3])
Dry run OK -- shapes check out.


## Train

In [15]:
from model.train import train

best_path, best_macro_f1 = train(
    epochs=18,
    batch_size=64,
    lr=3e-4,
    weight_decay=1e-4,
    num_workers=4,
    patience=5,
)
print(best_path, best_macro_f1)

Training on device: cuda
Class weights (['real', 'edited', 'deepfake']): [0.8782012462615967, 1.5744757652282715, 0.8155430555343628]

Epoch 1/18 (36.1s) train_loss=0.8867 val_macro_f1=0.6965
       real: precision=0.791 recall=0.467
     edited: precision=0.505 recall=0.963
   deepfake: precision=0.881 recall=0.802
  -> saved /content/deepfake/checkpoints/epoch_01.pt
  -> new best macro-F1 0.6965, saved to /content/deepfake/checkpoints/best_model.pt

Epoch 2/18 (36.1s) train_loss=0.3135 val_macro_f1=0.8937
       real: precision=0.876 recall=0.849
     edited: precision=0.884 recall=0.938
   deepfake: precision=0.910 recall=0.907
  -> saved /content/deepfake/checkpoints/epoch_02.pt
  -> new best macro-F1 0.8937, saved to /content/deepfake/checkpoints/best_model.pt

Epoch 3/18 (36.5s) train_loss=0.0792 val_macro_f1=0.9079
       real: precision=0.848 recall=0.921
     edited: precision=0.942 recall=0.895
   deepfake: precision=0.949 recall=0.898
  -> saved /content/deepfake/checkpoints

## Eval
Macro-F1, per-class precision/recall/ROC-AUC, 3x3 confusion matrix, and the paired Grad-CAM + gate-weight explainability dump.

In [9]:
from config import CHECKPOINT_DIR
from model.eval import load_model, compute_metrics, print_report, explainability_dump
from model.dataset import get_dataloader, ForgeryDataset

eval_model = load_model(str(CHECKPOINT_DIR / "best_model.pt"))
val_loader = get_dataloader('val', batch_size=64, shuffle=False, num_workers=4)
metrics = compute_metrics(eval_model, val_loader)
print_report(metrics)

Macro-F1: 0.9079

     class  precision     recall  roc_auc_ovr
      real      0.848      0.921        0.962
    edited      0.942      0.895        0.990
  deepfake      0.949      0.898        0.976

Confusion matrix (rows=true, cols=pred):
              real    edited  deepfake
    real       268         8        15
  edited        17       145         0
deepfake        31         1       281

edited->deepfake confusions: 0  deepfake->edited confusions: 1  (the novel failure mode this 3-class setup introduces -- watch this cell, not just accuracy)

Mean gate contribution weights (per-branch share of the fused prediction):
  overall  : spatial=0.287, spectral=0.414, noise_residual=0.298
      real : spatial=0.294, spectral=0.365, noise_residual=0.342
    edited : spatial=0.270, spectral=0.367, noise_residual=0.363
  deepfake : spatial=0.291, spectral=0.485, noise_residual=0.225


In [14]:
val_dataset = ForgeryDataset('val')
explainability_results = explainability_dump(eval_model, val_dataset, samples_per_class=3)

/content/drive/MyDrive/deepfake/dataset/val/real/real_01612_00000.jpg [real]: gate={'spatial': 0.287, 'spectral': 0.333, 'noise_residual': 0.38}
/content/drive/MyDrive/deepfake/dataset/val/real/real_01158_00001.jpg [real]: gate={'spatial': 0.3, 'spectral': 0.326, 'noise_residual': 0.375}
/content/drive/MyDrive/deepfake/dataset/val/real/real_02315_00002.jpg [real]: gate={'spatial': 0.33, 'spectral': 0.364, 'noise_residual': 0.306}
/content/drive/MyDrive/deepfake/dataset/val/edited/edited_Tp_S_NNN_S_N_cha10162_cha10162_12250_00000.jpg [edited]: gate={'spatial': 0.257, 'spectral': 0.374, 'noise_residual': 0.369}
/content/drive/MyDrive/deepfake/dataset/val/edited/edited_Tp_D_NNN_M_N_ani10132_ani10123_12477_00001.jpg [edited]: gate={'spatial': 0.279, 'spectral': 0.369, 'noise_residual': 0.352}
/content/drive/MyDrive/deepfake/dataset/val/edited/edited_Tp_D_NRN_S_B_sec00044_ind00020_00065_00002.jpg [edited]: gate={'spatial': 0.269, 'spectral': 0.386, 'noise_residual': 0.345}
/content/drive/My

In [20]:
# Visualize a few Grad-CAM heatmaps alongside their gate weights
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, min(4, len(explainability_results)), figsize=(16, 4))
for ax, r in zip(axes, explainability_results[:4]):
    img = Image.open(r['path']).convert('RGB')
    ax.imshow(img)
    ax.imshow(r['gradcam'], cmap='jet', alpha=0.4)
    gate_str = ', '.join(f"{k}:{v:.2f}" for k, v in r['gate_weights'].items())
    ax.set_title(f"{r['true_class']}\n{gate_str}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [19]:
!pip install -q gradio

from config import CHECKPOINT_DIR
from model.demo import build_app

checkpoint_path = str(globals().get('best_path', CHECKPOINT_DIR / 'best_model.pt'))
demo = build_app(checkpoint_path)
demo.launch(share=True)  # public *.gradio.live link, printed below

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://66c58dc5cf34716036.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
